In [1]:
# Install the dataretrieval package for accessing USGS water data
!pip install dataretrieval

In [2]:
# Defined by https://nvlpubs.nist.gov/nistpubs/Legacy/FIPS/fipspub5-2.pdf
# Standard FIPS state codes prefixed with "US:" for USGS/WQP REST API queries
code_to_state = {
    'US:01': 'Alabama',
    'US:02': 'Alaska',
    'US:04': 'Arizona',
    'US:05': 'Arkansas',
    'US:06': 'California',
    'US:08': 'Colorado',
    'US:09': 'Connecticut',
    'US:10': 'Delaware',
    'US:11': 'District of Columbia',
    'US:12': 'Florida',
    'US:13': 'Georgia',
    'US:15': 'Hawaii',
    'US:16': 'Idaho',
    'US:17': 'Illinois',
    'US:18': 'Indiana',
    'US:19': 'Iowa',
    'US:20': 'Kansas',
    'US:21': 'Kentucky',
    'US:22': 'Louisiana',
    'US:23': 'Maine',
    'US:24': 'Maryland',
    'US:25': 'Massachusetts',
    'US:26': 'Michigan',
    'US:27': 'Minnesota',
    'US:28': 'Mississippi',
    'US:29': 'Missouri',
    'US:30': 'Montana',
    'US:31': 'Nebraska',
    'US:32': 'Nevada',
    'US:33': 'New Hampshire',
    'US:34': 'New Jersey',
    'US:35': 'New Mexico',
    'US:36': 'New York',
    'US:37': 'North Carolina',
    'US:38': 'North Dakota',
    'US:39': 'Ohio',
    'US:40': 'Oklahoma',
    'US:41': 'Oregon',
    'US:42': 'Pennsylvania',
    'US:44': 'Rhode Island',
    'US:45': 'South Carolina',
    'US:46': 'South Dakota',
    'US:47': 'Tennessee',
    'US:48': 'Texas',
    'US:49': 'Utah',
    'US:50': 'Vermont',
    'US:51': 'Virginia',
    'US:53': 'Washington',
    'US:54': 'West Virginia',
    'US:55': 'Wisconsin',
    'US:56': 'Wyoming',
    # Key Territories
    'US:60': 'American Samoa',
    'US:66': 'Guam',
    'US:69': 'Northern Mariana Islands',
    'US:72': 'Puerto Rico',
    'US:78': 'Virgin Islands'
}

# Defined by https://nvlpubs.nist.gov/nistpubs/Legacy/FIPS/fipspub5-2.pdf
# Standard US State to FIPS code mapping for USGS/WQP REST API queries
state_to_code = {
    'Alabama': 'US:01',
    'Alaska': 'US:02',
    'Arizona': 'US:04',
    'Arkansas': 'US:05',
    'California': 'US:06',
    'Colorado': 'US:08',
    'Connecticut': 'US:09',
    'Delaware': 'US:10',
    'District of Columbia': 'US:11',
    'Florida': 'US:12',
    'Georgia': 'US:13',
    'Hawaii': 'US:15',
    'Idaho': 'US:16',
    'Illinois': 'US:17',
    'Indiana': 'US:18',
    'Iowa': 'US:19',
    'Kansas': 'US:20',
    'Kentucky': 'US:21',
    'Louisiana': 'US:22',
    'Maine': 'US:23',
    'Maryland': 'US:24',
    'Massachusetts': 'US:25',
    'Michigan': 'US:26',
    'Minnesota': 'US:27',
    'Mississippi': 'US:28',
    'Missouri': 'US:29',
    'Montana': 'US:30',
    'Nebraska': 'US:31',
    'Nevada': 'US:32',
    'New Hampshire': 'US:33',
    'New Jersey': 'US:34',
    'New Mexico': 'US:35',
    'New York': 'US:36',
    'North Carolina': 'US:37',
    'North Dakota': 'US:38',
    'Ohio': 'US:39',
    'Oklahoma': 'US:40',
    'Oregon': 'US:41',
    'Pennsylvania': 'US:42',
    'Rhode Island': 'US:44',
    'South Carolina': 'US:45',
    'South Dakota': 'US:46',
    'Tennessee': 'US:47',
    'Texas': 'US:48',
    'Utah': 'US:49',
    'Vermont': 'US:50',
    'Virginia': 'US:51',
    'Washington': 'US:53',
    'West Virginia': 'US:54',
    'Wisconsin': 'US:55',
    'Wyoming': 'US:56',
    # Key Territories
    'American Samoa': 'US:60',
    'Guam': 'US:66',
    'Northern Mariana Islands': 'US:69',
    'Puerto Rico': 'US:72',
    'Virgin Islands': 'US:78'
}

In [3]:
from dataretrieval import wqp

# Good lord I had forgotten how big of a pain it was to pull good data from the USGS. 
# 
# Let's pull ALL water temperature and pH records for Washington state (US:53) from 2015 to 2025.
# This will include all stream monitoring sites in the state, and all records of those two parameters over 
# for a 10-year span. This will yield a massive, messy dataset.

# The WQP API is pretty flexible and allows you to specify a wide range of parameters to filter the results.
# The dataretrieval package provides a convenient interface to access this data directly from Python, and it
# will return the results as a pandas DataFrame, which is great for analysis and manipulation, but its kind
# of a pain to figure out the right parameters to pass to get the data you want. 
# The documentation is pretty sparse, and the parameter names are not always intuitive.  
# https://www.waterqualitydata.us/webservices_documentation/

# Start with WA
state_name = "Washington"
fips_code = state_to_code.get(state_name)

site_type = 'Stream' # We want stream monitoring sites, not lakes or groundwater wells
characteristics = ['Temperature, water', 'pH'] # The parameters we want to analyze,  The names are derived from the USEPA Substance Registry System.
start_date = '2015-01-01'
end_date = '2025-01-01'

print("Fetching stream monitoring sites in Washington...")

# Step 1: Find the stream sites in the given state
sites = wqp.what_sites(statecode=fips_code, siteType=site_type)
print(f"Found {len(sites)} stream monitoring sites.")

# Okay, so the sites variables is a pandas Dataframe and a "dataretrieval.wqp.WQP_Metadata" object.
# The WQP_Metadata object contains the metadata about the query, and the DataFrame contains the actual data.

sites_df = sites[0] # The first item in the tuple is the DataFrame
metadata = sites[1] # The second item in the tuple is the metadata

print(f"sites_df columns: {sites_df.columns}")

print("Stream monitoring sites in Washington:")
print(sites_df.head())
print ("...")
print(sites_df.tail())


print(f"Metadata: {metadata}")

# results = wqp.get_results(
#     statecode=state_code, 
#     characteristicName=characteristics,
#     startDateLo=start_date,
#     startDateHi=end_date
# )

# # Convert to pandas
# df = results[0] # The first item in the tuple is the DataFrame
# print(f"Downloaded {len(df)} rows of raw water quality data!")

Fetching stream monitoring sites in Washington...
Found 2 stream monitoring sites.
sites_df columns: Index(['OrganizationIdentifier', 'OrganizationFormalName',
       'MonitoringLocationIdentifier', 'MonitoringLocationName',
       'MonitoringLocationTypeName', 'MonitoringLocationDescriptionText',
       'HUCEightDigitCode', 'DrainageAreaMeasure/MeasureValue',
       'DrainageAreaMeasure/MeasureUnitCode',
       'ContributingDrainageAreaMeasure/MeasureValue',
       'ContributingDrainageAreaMeasure/MeasureUnitCode', 'LatitudeMeasure',
       'LongitudeMeasure', 'SourceMapScaleNumeric',
       'HorizontalAccuracyMeasure/MeasureValue',
       'HorizontalAccuracyMeasure/MeasureUnitCode',
       'HorizontalCollectionMethodName',
       'HorizontalCoordinateReferenceSystemDatumName',
       'VerticalMeasure/MeasureValue', 'VerticalMeasure/MeasureUnitCode',
       'VerticalAccuracyMeasure/MeasureValue',
       'VerticalAccuracyMeasure/MeasureUnitCode',
       'VerticalCollectionMethodName',


In [4]:
# Hmm. The data is pretty messy. There are a lot of columns, and many of them are not relevant to our analysis.
# Lots of NaN values, and ProviderName column contains both NWIS (USGS National Water Information System) and STORET (EPA’s Storage and Retrieval system).
# Welp, time to clean it up. 

class ValidationError(Exception):
    """Custom exception for validation errors. We'll expand this later, but for now it's just a placeholder."""
    pass

from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime, timezone

class MonitoringSiteSchema(BaseModel):
    # Core identifiers
    org_id: str = Field(..., alias="OrganizationIdentifier")
    org_name: str = Field(..., alias="OrganizationFormalName")
    site_id: str = Field(..., alias="MonitoringLocationIdentifier")
    site_name: str = Field(..., alias="MonitoringLocationName")
    site_type: str = Field(..., alias="MonitoringLocationTypeName")
    
    # Geographic data (coerced automatically to floats)
    latitude: float = Field(..., alias="LatitudeMeasure")
    longitude: float = Field(..., alias="LongitudeMeasure")
    
    # Standardized Codes (We accept coerced strings and clean up decimal artifacts)
    state_code: str = Field(..., alias="StateCode")
    
    # There are a few missing HUC and county codes, but since we have lat/lon we can always derive those
    # later if needed. So we make these optional and allow them to be None if missing.
    # huc_8: Optional[str] = Field(None, alias="HUCEightDigitCode")
    # county_code: Optional[str] = Field(None, alias="CountyCode")
    
    huc_8: str = Field(..., alias="HUCEightDigitCode")
    county_code: str = Field(..., alias="CountyCode")
    
    # Optional metadata (will default to None if missing/NaN in the data)
    drainage_area: Optional[float] = Field(None, alias="DrainageAreaMeasure/MeasureValue")
    drainage_unit: Optional[str] = Field(None, alias="DrainageAreaMeasure/MeasureUnitCode")
    provider: str = Field(..., alias="ProviderName")

    # Clean up numeric codes that Pandas converted to float strings (e.g., '17010214.0' -> '17010214')
    @field_validator("huc_8", "county_code", "state_code", mode="before")
    @classmethod
    def clean_numeric_code_strings(cls, v) -> str:
        if v is None:
            return v
        
        # Coerce to a raw string if it is an int/float
        val_str = str(v).strip()
        
        # If it ends in '.0' (Pandas float inference artifact), strip it
        if val_str.endswith(".0"):
            val_str = val_str[:-2]
            
        return val_str
    
    # Some site names are missing or just whitespace. Let's replace those with a standard placeholder to avoid issues downstream.
    @field_validator("site_name", mode="before")
    @classmethod
    def heal_empty_site_names(cls, v) -> str:
        # If the name is None, empty, or pandas NaN, replace it with a fallback
        if v is None or str(v).strip() == "" or v is np.nan:
            return "UNKNOWN LOCATION NAME"
        return str(v).strip()

    # Coordinate validation to ensure lat/lon are within valid ranges
    @field_validator("latitude")
    @classmethod
    def validate_latitude(cls, v: float) -> float:
        if not (-90.0 <= v <= 90.0):
            raise ValueError(f"Latitude {v} is out of bounds.")
        return v

    @field_validator("longitude")
    @classmethod
    def validate_longitude(cls, v: float) -> float:
        if not (-180.0 <= v <= 180.0):
            raise ValueError(f"Longitude {v} is out of bounds.")
        return v
    
    # have to set coercion to string for these code fields because WQP uses strings to represent codes (to preserve leading zeros), but the raw data may have them as numbers, which causes validation errors.
    # CountyCode
    #   Input should be a valid string [type=string_type, input_value=63.0, input_type=float]
        
    model_config = {
        "populate_by_name": True, # Allows instantiating with either the field name or its alias
        "arbitrary_types_allowed": True,
        "coerce_numbers_to_str": True 
    }

In [5]:
# Create local directory structure if they don't exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/bronze", exist_ok=True)
os.makedirs("data/quarantine", exist_ok=True)

# 1. Save original raw DataFrame locally as CSV in raw landing zone first
raw_path = f"data/raw/{state_name.lower().replace(' ', '_')}_stream_sites.csv"
sites_df.to_csv(raw_path, index=False)
print(f"Saved raw landing file to: {raw_path}")

# Initialize our tracking lists and telemetry counters
validated_records = []
quarantined_records = []

# Convert DataFrame to list of dicts to process. 
# Replacing NaN with None makes it compatible with Pydantic's Optional types.
raw_records = sites_df.replace({np.nan: None}).to_dict(orient="records")

print(f"Starting validation on {len(raw_records)} records...")
start_time = datetime.now()

for idx, record in enumerate(raw_records):
    try:
        # Pass the raw record dict directly to Pydantic
        validated_model = MonitoringSiteSchema(**record)
        
        # Serialize back to dictionary using model_dump() (using real field names, not aliases)
        clean_dict = validated_model.model_dump()
        
        # Add technical metadata for Bronze layer
        clean_dict["ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
        
        validated_records.append(clean_dict)
        
    except ValidationError as e:
        # capture the exact failure telemetry
        error_details = e.errors()
        
        quarantined_entry = {
            "index": idx,
            "raw_record": record,
            "validation_errors": [
                {
                    "field": " -> ".join(map(str, err["loc"])),
                    "error_message": err["msg"],
                    "type": err["type"]
                }
                for err in error_details
            ],
            "quarantined_at": datetime.now(tz=timezone.utc).isoformat()
        }
        quarantined_records.append(quarantined_entry)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Print pipeline telemetry metrics
print("\n--- INGESTION TELEMETRY METRICS ---")
print(f"Processing Time   : {duration:.2f} seconds")
print(f"Total Processed   : {len(raw_records)}")
print(f"Passed validation : {len(validated_records)} ({len(validated_records)/len(raw_records)*100:.2f}%)")
print(f"Quarantined (Bad) : {len(quarantined_records)} ({len(quarantined_records)/len(raw_records)*100:.2f}%)")
print("----------------------------------")

# Write out the validated records to the Bronze layer as Parquet
bronze_path = f"data/bronze/{state_name.lower().replace(' ', '_')}_stream_sites_validated.parquet"
pd.DataFrame(validated_records).to_parquet(bronze_path, index=False)
print(f"Saved validated records to Bronze layer: {bronze_path}")

# Write out the quarantined records to a JSON file for later analysis
quarantine_path = f"data/quarantine/{state_name.lower().replace(' ', '_')}_stream_sites_quarantined.json"
with open(quarantine_path, "w") as f:
    json.dump(quarantined_records, f, indent=4)
print(f"Saved quarantined records to: {quarantine_path}")


Saved raw landing file to: data/raw/washington_stream_sites.csv
Starting validation on 17936 records...

--- INGESTION TELEMETRY METRICS ---
Processing Time   : 0.12 seconds
Total Processed   : 17936
Passed validation : 17927 (99.95%)
Quarantined (Bad) : 9 (0.05%)
----------------------------------
Saved validated records to Bronze layer: data/bronze/washington_stream_sites_validated.parquet
Saved quarantined records to: data/quarantine/washington_stream_sites_quarantined.json


## Quarantine Zone

What went wrong? Lets figure it out. 

In [6]:
import json
import pandas as pd

# Load the quarantined records
with open(quarantine_path, "r") as f:
    bad_records = json.load(f)

# Flatten the nested errors into a Pandas DataFrame for easy analysis
error_summary = []
for entry in bad_records:
    for err in entry["validation_errors"]:
        error_summary.append({
            "site_id": entry["raw_record"].get("MonitoringLocationIdentifier"),
            "failed_field": err["field"],
            "error_msg": err["error_message"],
            "raw_value": entry["raw_record"].get(err["field"].split(" -> ")[-1])
        })

err_df = pd.DataFrame(error_summary)

print(f"Total Unique Errors: {len(err_df)}")
print("\n--- ERROR TYPE BREAKDOWN ---")
print(err_df["failed_field"].value_counts())

print("\n--- SAMPLE ERROR ENTRIES ---")
for unique_field in err_df["failed_field"].unique():
    print(f"\nErrors for field: {unique_field}")
    sample_errors = err_df[err_df["failed_field"] == unique_field].iloc[:1]  # Get the first error for this field
    print(sample_errors[["site_id", "error_msg", "raw_value"]])


Total Unique Errors: 18

--- ERROR TYPE BREAKDOWN ---
failed_field
LatitudeMeasure      7
LongitudeMeasure     7
HUCEightDigitCode    2
CountyCode           2
Name: count, dtype: int64

--- SAMPLE ERROR ENTRIES ---

Errors for field: LatitudeMeasure
           site_id                       error_msg raw_value
0  USGS-1203950350  Input should be a valid number      None

Errors for field: LongitudeMeasure
           site_id                       error_msg raw_value
1  USGS-1203950350  Input should be a valid number      None

Errors for field: HUCEightDigitCode
             site_id                       error_msg raw_value
14  LUMMINSN_WQX-T54  Input should be a valid string      None

Errors for field: CountyCode
             site_id                       error_msg raw_value
15  LUMMINSN_WQX-T54  Input should be a valid string      None


## Best Effort 

First pass we had:

```
Total Unique Errors: 206
--- ERROR TYPE BREAKDOWN ---
failed_field 
MonitoringLocationName 188
LatitudeMeasure 7
LongitudeMeasure 7
HUCEightDigitCode 2
CountyCode 2
``` 

### MonitoringLocationName

It is okay for this to be blank. Adjusted to handling missing values as UNKNOWN_LOCATION. Accept and Postpone.

### Latitude/Longitude

We cannot accept records with no coordinates, only a description of "BOULDER CREEK NEAR MOUTH NEAR AMANDA PARK, WA". REJECT.

### HUC-8 Codes

If your [HUC-8 code](https://nas.er.usgs.gov/hucs.aspx) is missing, it should be relatively easy to backfill it later with some Pro-GIS Engineer work. Accept and Postpone.

### County Code

Same deal as the HUC-8 codes. A real Pro-GIS maverick is gonna love handling that. Accept and Postpone.

There  I marked them optional, with a default value of None. Since we have the rest of the data, we can backfill if its worth it for the few that are missing.

In [7]:
# hahahah postpone...
!pip install geopy

import pandas as pd

# Create a mapping dictionary for state names to 2-letter state abbreviations
# (Census uses 2-letter state abbreviations like 'WA', 'AL' in the 'State' column)
state_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}

def normalize_name(name_str):
    """ Normalize county names by lowercasing, stripping whitespace, and removing common administrative terms."""
    if not name_str:
        return ""

    normalized = str(name_str).lower().strip()
    normalized = normalized.replace("county", "").replace("parish", "").strip()
    return normalized

# Load the FIPS table
url = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
fips_df = pd.read_csv(url, header=None, names=["State", "StateFIPS", "CountyFIPS", "CountyName", "CLASSFP"], dtype=str)
fips_df['FIPS'] = fips_df['StateFIPS'] + fips_df['CountyFIPS']

fips_df['normalized_county'] = fips_df['CountyName'].apply(normalize_name)
fips_df['normalized_state'] = fips_df['State'].apply(lambda x: str(x).lower().strip())


# Apply normalization for matching keys
fips_df['normalized_county'] = fips_df['CountyName'].apply(normalize_name)
fips_df['normalized_state'] = fips_df['State'].apply(lambda x: str(x).lower().strip())

# Save this cleaned lookup index locally
fips_df.to_csv("data/raw/fips_lookup_clean.csv", index=False)


In [8]:

def resolve_fips(raw_state, raw_county, lookup_df=fips_df):
    """
    Takes full/abbreviated state name and raw county name, 
    normalizes them, and extracts the 5-digit FIPS code.
    """
    if not raw_state or not raw_county:
        return None
        
    # Get the 2-letter state abbreviation if full name is provided
    state_abbr = state_to_abbr.get(raw_state, raw_state).lower().strip()
    norm_county = normalize_name(raw_county)
    
    # Query the normalized FIPS table
    match = lookup_df[
        (lookup_df['normalized_state'] == state_abbr) & 
        (lookup_df['normalized_county'] == norm_county)
    ]
    
    if not match.empty:
        # Return the 5-digit FIPS code
        return match['FIPS'].values[0]
    return None

# Example usage:
fips_code = resolve_fips("Washington", "King County")
print(f"Resolved FIPS code: {fips_code} == 53033")

# and for skagit county, WA
fips_code = resolve_fips("Washington", "Skagit County")
print(f"Resolved FIPS code: {fips_code} == 53057")

Resolved FIPS code: 53033 == 53033
Resolved FIPS code: 53057 == 53057


In [ ]:
# TODO: use the USGS api to pull the huc_8 code from the lat/lon if we can't resolve the county/state from the raw data, since we have the coordinates for all the sites.

from geopy.geocoders import Nominatim
import us # for getting the fips code for the state, once we have the county name, which we can derive from the lat/lon if needed.
import time


# Found a tutorial for the geocoders bit: https://www.geeksforgeeks.org/python/get-the-city-state-and-country-names-from-latitude-and-longitude-using-python/
# Second part is using the us package to get the FIPS code for the state, once we have the county name, which we can derive from the lat/lon if needed. 

# initialize Nominatim API 
geolocator = Nominatim(user_agent="county_finder")
successful_backfills = []

# Rate-limit Nominatim requests to avoid hitting block limits (1 second per request is polite)
print(f"Starting spatial backfill process for {len(quarantined_records)} records...\n")

for bad_record in quarantined_records:
    site_id = bad_record['raw_record'].get('MonitoringLocationIdentifier')
    
    lat = bad_record['raw_record'].get('LatitudeMeasure')
    lon = bad_record['raw_record'].get('LongitudeMeasure')
    

    if lat is None or lon is None:
        print(f"[Unrecoverable] Site {site_id} is missing coordinates. Cannot backfill.")
        continue
    
    # I need lon west, so I need to make the longitude negative.
    lon = -lon if lon > 0 else lon

    try:
        print(f"Geocoding Site {site_id} ({lat}, {lon})...")
        location = geolocator.reverse(f"{lat}, {lon}", exactly_one=True)
        
        # Example: Skagit County, Washington, United States
        # We can parse the location.raw['address'] dictionary to extract the county, state, and country information.
        
        if location and 'address' in location.raw:
            address = location.raw['address']
            county_name = address.get('county')
            state_name = address.get('state')
            
            # Use our standardized matching technique to resolve the FIPS code from the derived county and state names
            fips_5_digit = resolve_fips(state_name, county_name)
            
            if fips_5_digit:
                # WQP separates FIPS into state_code (2 digits) and county_code (3 digits)
                derived_state_code = fips_5_digit[:2]
                derived_county_code = fips_5_digit[2:]
                
                successful_backfills.append({
                    "site_id": site_id,
                    "site_name": bad_record['raw_record'].get('MonitoringLocationName') or "UNKNOWN LOCATION NAME",
                    "lat": lat,
                    "lon": lon,
                    "county": county_name,
                    "state": state_name,
                    "state_code": derived_state_code,
                    "county_code": derived_county_code,
                    "huc_8": "UNKNOWN",  # Or handle watersheds down the road
                    "source_record": bad_record['raw_record']
                })
                print(f"Successfully resolved FIPS: {fips_5_digit} (State: {derived_state_code}, County: {derived_county_code})")
            else:
                print(f"Could not resolve FIPS for State: {state_name}, County: {county_name}")
        else:
            print(f"Geocoding returned no address data for {site_id}")
            
        # Nominatim asks for max 1 request per second in their Terms of Service
        time.sleep(1.0)
    except Exception as e:
            print(f"Error geocoding site {site_id}: {e}")
            time.sleep(1.0)

    print(f"\nCompleted! Successfully backfilled {len(successful_backfills)} out of {len(quarantined_records)} quarantined records.")

for backfill in successful_backfills:
    print(f"Backfilled Site ID: {backfill['site_id']}")
    print(f"Derived County: {backfill['county']}")
    print(f"Derived State: {backfill['state']}")
    print(f"Derived County Code: {backfill['county_code']}")
    print(f"Derived HUC-8 Code: {backfill['huc_8']}")
    print("\n")


Starting spatial backfill process for 9 records...

[Unrecoverable] Site USGS-1203950350 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-1203950370 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-1203950790 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-1203950980 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-1203951185 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-1203951475 is missing coordinates. Cannot backfill.
[Unrecoverable] Site USGS-12062512 is missing coordinates. Cannot backfill.
Geocoding Site LUMMINSN_WQX-T54 (48.61986, -122.10263)...
Successfully resolved FIPS: 53057 (State: 53, County: 057)

Completed! Successfully backfilled 1 out of 9 quarantined records.
Geocoding Site LUMMINSN_WQX-T59 (48.61829, -122.10245)...
Successfully resolved FIPS: 53057 (State: 53, County: 057)

Completed! Successfully backfilled 2 out of 9 quarantined records.
Backfilled Site ID: LUMMINSN_WQX-T